# ForensicNet — TGIF Subset Image Forgery Detection

**Model:** ForensicNet (ConvNeXt-Large + SRM/Bayar noise branch + FPN decoder)  
**All hyperparameters, loss, metrics, scheduler, checkpointing — identical to SegFormer baseline.**  
**Augmentation:** OFF (fair comparison)

In [2]:
# ── Cell 1: Install ───────────────────────────────────────────────────────────
!pip install timm albumentations tqdm -q

In [3]:
!kaggle datasets download -d araftahsanpavel/tgif-subset -p /kaggle/working/

import zipfile
print("Extracting...")
with zipfile.ZipFile('/kaggle/working/tgif-subset.zip', 'r') as z:
    z.extractall('/kaggle/working/')
print("✓ Done")

Dataset URL: https://www.kaggle.com/datasets/araftahsanpavel/tgif-subset
License(s): unknown
100%|███████████████████████████████████████| 8.88G/8.88G [01:03<00:00, 149MB/s]

Extracting...
✓ Done


In [4]:
# ── Cell 2: Imports & Config (IDENTICAL hyperparams to SegFormer baseline) ────
from __future__ import annotations
import json, os, random, sys, time, traceback
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.amp import autocast, GradScaler

import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm.auto import tqdm
import timm

# ==============================================================================
# HYPERPARAMETERS — IDENTICAL to SegFormer baseline (DO NOT CHANGE)
# ==============================================================================
IMAGE_SIZE          = 384
EPOCHS              = 30
BATCH_SIZE          = 16
LEARNING_RATE       = 1e-4
WEIGHT_DECAY        = 1e-4
LR_PATIENCE         = 3
LR_FACTOR           = 0.5
LR_MIN              = 1e-6
EARLY_STOP_PATIENCE = 7
BINARY_THRESHOLD    = 0.5
GRAD_CLIP_NORM      = 1.0
MIXED_PRECISION     = True
SEED                = 42
NUM_WORKERS         = 4
PIN_MEMORY          = True
POS_WEIGHT          = 3.0

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

DATA_ROOT  = Path("/kaggle/input/tgif-subset")
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print("  ForensicNet (ConvNeXt-L) — TGIF subset binary forgery localization")
print("=" * 70)
if device.type == "cuda":
    print(f"  GPU        : {torch.cuda.get_device_name(0)}  "
          f"({torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB)")
print(f"  POS_WEIGHT : {POS_WEIGHT}")
print(f"  IMAGE_SIZE : {IMAGE_SIZE}")
print(f"  EPOCHS     : {EPOCHS}")
print(f"  BATCH_SIZE : {BATCH_SIZE}")
print("=" * 70)

  ForensicNet (ConvNeXt-L) — TGIF subset binary forgery localization
  GPU        : Tesla T4  (14.6 GB)
  POS_WEIGHT : 3.0
  IMAGE_SIZE : 384
  EPOCHS     : 30
  BATCH_SIZE : 16


In [5]:
# ── Cell 3: Reproducibility (IDENTICAL) ──────────────────────────────────────
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def get_rng_state() -> Dict[str, Any]:
    return {
        "python":     random.getstate(),
        "numpy":      np.random.get_state(),
        "torch":      torch.get_rng_state(),
        "torch_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }

def set_rng_state(state: Dict[str, Any]) -> None:
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if state.get("torch_cuda") is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(state["torch_cuda"])

set_seed(SEED)
print("OK")

OK


In [6]:
# ── Cell 4: Dataset Indexing (IDENTICAL) ─────────────────────────────────────
DATA_ROOT = Path("/kaggle/working/subset")
def build_index(split_dir: Path) -> List[Dict[str, Any]]:
    if not split_dir.exists():
        raise FileNotFoundError(f"Split folder not found: {split_dir}")

    images_root = split_dir / "images"
    masks_root  = split_dir / "masks"
    fakes_root  = split_dir / "fakes"
    samples: List[Dict[str, Any]] = []

    if images_root.exists():
        for p in images_root.rglob("*_orig.png"):
            samples.append({"image_path": str(p), "mask_path": None,
                             "is_fake": False, "category": p.parent.name})

    if fakes_root.exists():
        for p in fakes_root.rglob("*_sd2_*.png"):
            name = p.name
            if "_mask_" not in name:
                continue
            coco_id, rest = name.split("_mask_", 1)
            mask_type = rest.split(".png", 1)[0]
            category  = p.parent.name
            paired_mask = (
                masks_root / category
                / f"{coco_id}_mask_{mask_type}.png_ps_mask.png"
            )
            if not paired_mask.exists():
                continue
            samples.append({"image_path": str(p), "mask_path": str(paired_mask),
                             "is_fake": True, "category": category})
    return samples


print("[1/5] Indexing dataset ...")
train_samples = build_index(DATA_ROOT / "training")
val_samples   = build_index(DATA_ROOT / "validation")
test_samples  = build_index(DATA_ROOT / "testing")

for name, samples in [("training", train_samples),
                       ("validation", val_samples),
                       ("testing", test_samples)]:
    n_fake = sum(1 for s in samples if s["is_fake"])
    n_real = len(samples) - n_fake
    print(f"   {name:<10}: {len(samples):>5} total  ({n_real:>4} real, {n_fake:>4} fake)")

[1/5] Indexing dataset ...
   training  :  7559 total  (2100 real, 5459 fake)
   validation:  1023 total  ( 341 real,  682 fake)
   testing   :  1029 total  ( 343 real,  686 fake)


In [7]:
# ── Cell 5: Dataset Class (IDENTICAL — augment=False everywhere) ──────────────
class TGIFDataset(Dataset):
    def __init__(self, samples: List[Dict[str, Any]], image_size: int, augment: bool) -> None:
        self.samples    = samples
        self.image_size = image_size
        self.augment    = augment

        # augment branch kept for code parity — always called with augment=False
        if augment:
            self.transform = A.Compose([
                A.Resize(image_size, image_size, interpolation=cv2.INTER_LINEAR),
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.3),
                A.RandomRotate90(p=0.3),
                A.ColorJitter(brightness=0.2, contrast=0.2,
                              saturation=0.1, hue=0.0, p=0.4),
                A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
                ToTensorV2(),
            ])
        else:
            self.transform = A.Compose([
                A.Resize(image_size, image_size, interpolation=cv2.INTER_LINEAR),
                A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
                ToTensorV2(),
            ])

    def __len__(self) -> int:
        return len(self.samples)

    def _read_image(self, path: str) -> np.ndarray:
        img = cv2.imread(path, cv2.IMREAD_COLOR)
        if img is None:
            raise RuntimeError(f"Could not read image: {path}")
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    def _read_mask(self, path: Optional[str], h: int, w: int) -> np.ndarray:
        if path is None:
            return np.zeros((h, w), dtype=np.uint8)
        m = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if m is None:
            raise RuntimeError(f"Could not read mask: {path}")
        return (m >= 127).astype(np.uint8)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        s    = self.samples[idx]
        img  = self._read_image(s["image_path"])
        mask = self._read_mask(s["mask_path"], img.shape[0], img.shape[1])
        out  = self.transform(image=img, mask=mask)
        return {
            "image":   out["image"].float(),
            "mask":    out["mask"].float().unsqueeze(0),
            "is_fake": torch.tensor(s["is_fake"], dtype=torch.bool),
        }


print("[2/5] Building DataLoaders (augment=False for all splits) ...")
train_loader = DataLoader(
    TGIFDataset(train_samples, IMAGE_SIZE, augment=False),   # no augmentation
    batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
)
val_loader = DataLoader(
    TGIFDataset(val_samples, IMAGE_SIZE, augment=False),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
)
test_loader = DataLoader(
    TGIFDataset(test_samples, IMAGE_SIZE, augment=False),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
)
print(f"   Train batches : {len(train_loader)}")
print(f"   Val batches   : {len(val_loader)}")
print(f"   Test batches  : {len(test_loader)}")

[2/5] Building DataLoaders (augment=False for all splits) ...
   Train batches : 473
   Val batches   : 64
   Test batches  : 65


In [8]:
# ── Cell 6: Loss & Metrics (IDENTICAL) ───────────────────────────────────────
class DiceLoss(nn.Module):
    def __init__(self, smooth: float = 1.0) -> None:
        super().__init__()
        self.smooth = smooth

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs   = torch.sigmoid(logits).view(logits.size(0), -1)
        targets = targets.view(targets.size(0), -1)
        inter   = (probs * targets).sum(dim=1)
        denom   = probs.sum(dim=1) + targets.sum(dim=1)
        return 1.0 - ((2.0 * inter + self.smooth) / (denom + self.smooth)).mean()


class BCEDiceLoss(nn.Module):
    def __init__(self, pos_weight: float = POS_WEIGHT) -> None:
        super().__init__()
        self.register_buffer("pw", torch.tensor([pos_weight]))
        self.dice = DiceLoss()

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        bce = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=self.pw)
        return 0.5 * bce + 0.5 * self.dice(logits, targets)


class MetricAccumulator:
    def __init__(self) -> None:
        self.tp = self.fp = self.tn = self.fn = 0

    def update(self, preds_binary: torch.Tensor, targets: torch.Tensor) -> None:
        p = preds_binary.bool(); t = targets.bool()
        self.tp += int(( p &  t).sum().item())
        self.fp += int(( p & ~t).sum().item())
        self.tn += int((~p & ~t).sum().item())
        self.fn += int((~p &  t).sum().item())

    def compute(self) -> Dict[str, float]:
        eps = 1e-8
        tp, fp, tn, fn = self.tp, self.fp, self.tn, self.fn
        precision = tp / (tp + fp + eps)
        recall    = tp / (tp + fn + eps)
        f1        = 2 * precision * recall / (precision + recall + eps)
        return {
            "pixel_accuracy": (tp + tn) / (tp + fp + tn + fn + eps),
            "precision":      precision,
            "recall":         recall,
            "f1":             f1,
            "iou":            tp / (tp + fp + fn + eps),
            "dice":           2 * tp / (2 * tp + fp + fn + eps),
            "tp": tp, "fp": fp, "tn": tn, "fn": fn,
        }

print("OK")

OK


In [9]:
# ── Cell 7: ForensicNet Architecture ─────────────────────────────────────────
# ConvNeXt-Large (ImageNet-22k) + SRM + BayarConv + FPN + CBAM
# Output shape: (B, 1, IMAGE_SIZE, IMAGE_SIZE) — drop-in for SegFormerBinary

class SRMConv(nn.Module):
    """3 fixed SRM high-pass kernels applied to each RGB channel."""
    def __init__(self):
        super().__init__()
        f1 = np.array([[0, 0, 0, 0, 0],[0,-1, 2,-1, 0],[0, 2,-4, 2, 0],
                       [0,-1, 2,-1, 0],[0, 0, 0, 0, 0]], dtype=np.float32) / 4.0
        f2 = np.array([[-1, 2,-2, 2,-1],[ 2,-6, 8,-6, 2],[-2, 8,-12,8,-2],
                       [ 2,-6, 8,-6, 2],[-1, 2,-2, 2,-1]], dtype=np.float32) / 12.0
        f3 = np.array([[0, 0, 0, 0, 0],[0, 0, 0, 0, 0],[0, 1,-2, 1, 0],
                       [0, 0, 0, 0, 0],[0, 0, 0, 0, 0]], dtype=np.float32) / 2.0
        self.register_buffer('weight', torch.from_numpy(np.stack([f1,f2,f3])[:,None]))

    def forward(self, x):
        return torch.cat([F.conv2d(x[:,c:c+1], self.weight, padding=2) for c in range(3)], dim=1)


class BayarConv2d(nn.Module):
    """Constrained conv — center weight forced to -(sum of neighbors)."""
    def __init__(self, in_ch=3, out_ch=3, k=5):
        super().__init__()
        self.k = k
        self.weight = nn.Parameter(torch.randn(out_ch, in_ch, k, k) * 0.01)

    def forward(self, x):
        w, c = self.weight.clone(), self.k // 2
        wn = w.clone(); wn[:,:,c,c] = 0.0
        w[:,:,c,c] = -wn.sum(dim=[2,3])
        return F.conv2d(x, w, padding=c)


class CBAM(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__()
        mid = max(ch // r, 8)
        self.ca = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                nn.Linear(ch, mid), nn.ReLU(),
                                nn.Linear(mid, ch), nn.Sigmoid())
        self.sa = nn.Sequential(nn.Conv2d(2, 1, 7, padding=3, bias=False), nn.Sigmoid())

    def forward(self, x):
        x = x * self.ca(x).view(x.size(0), -1, 1, 1)
        return x * self.sa(torch.cat([x.mean(1,keepdim=True), x.amax(1,keepdim=True)], 1))


class FPNBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up   = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.GELU(),
        )
        self.cbam = CBAM(out_ch)

    def forward(self, x, skip=None):
        x = self.up(x)
        if skip is not None:
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=True)
            x = torch.cat([x, skip], dim=1)
        return self.cbam(self.conv(x))


class NoiseBranch(nn.Module):
    def __init__(self, out_ch=128):
        super().__init__()
        self.srm   = SRMConv()
        self.bayar = BayarConv2d(3, 3)
        self.enc   = nn.Sequential(
            nn.Conv2d(12, 32, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(64), nn.GELU(),
            nn.Conv2d(64, out_ch, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.GELU(),
        )

    def forward(self, x):
        return self.enc(torch.cat([self.srm(x), self.bayar(x)], dim=1))


class ForensicNet(nn.Module):
    """
    ConvNeXt-Large (ImageNet-22k) + SRM/Bayar noise branch + FPN decoder.
    Output: (B, 1, image_size, image_size) — drop-in for SegFormerBinary.
    ConvNeXt-L stage channels: [192, 384, 768, 1536]
    """
    def __init__(self, image_size: int = IMAGE_SIZE, pretrained: bool = True):
        super().__init__()
        self.image_size = image_size
        self.enc   = timm.create_model('convnext_large', pretrained=pretrained,
                                        features_only=True, out_indices=(0,1,2,3))
        self.noise = NoiseBranch(out_ch=128)
        self.lat4  = nn.Conv2d(1536+128, 256, 1)
        self.lat3  = nn.Conv2d(768,      256, 1)
        self.lat2  = nn.Conv2d(384,      128, 1)
        self.lat1  = nn.Conv2d(192,       64, 1)
        self.dec3  = FPNBlock(256, 256, 256)
        self.dec2  = FPNBlock(256, 128, 128)
        self.dec1  = FPNBlock(128,  64,  64)
        self.dec0  = FPNBlock( 64,   0,  32)
        self.head  = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(32, 16, 3, padding=1, bias=False), nn.BatchNorm2d(16), nn.GELU(),
            nn.Conv2d(16,  1, 1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        s1, s2, s3, s4 = self.enc(x)                            # H/4, H/8, H/16, H/32
        n_s4 = F.adaptive_avg_pool2d(self.noise(x), s4.shape[2:])  # 128ch @ H/32
        p4 = self.lat4(torch.cat([s4, n_s4], dim=1))
        d3 = self.dec3(p4,          self.lat3(s3))
        d2 = self.dec2(d3,          self.lat2(s2))
        d1 = self.dec1(d2,          self.lat1(s1))
        d0 = self.dec0(d1)
        out = self.head(d0)
        if out.shape[2:] != (self.image_size, self.image_size):
            out = F.interpolate(out, size=(self.image_size, self.image_size),
                                mode='bilinear', align_corners=False)
        return out


# Sanity check
with torch.no_grad():
    _m = ForensicNet(pretrained=False)
    _o = _m(torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE))
    assert _o.shape == (2, 1, IMAGE_SIZE, IMAGE_SIZE), f"Bad shape: {_o.shape}"
    total = sum(p.numel() for p in _m.parameters())
    print(f"OK  output={_o.shape}  params={total/1e6:.1f}M  size={total*4/1024**2:.0f}MB")
    del _m, _o

OK  output=torch.Size([2, 1, 384, 384])  params=199.6M  size=761MB


In [10]:
# ── Cell 8: Training Loop (IDENTICAL) ────────────────────────────────────────
def run_one_epoch(
    model, loader, criterion, optimizer, scaler,
    device, train, desc, print_mem_at_batch_1=False,
):
    model.train(mode=train)
    total_loss, n_batches = 0.0, 0
    metrics  = MetricAccumulator()
    grad_ctx = torch.enable_grad() if train else torch.no_grad()

    with grad_ctx:
        pbar = tqdm(loader, desc=desc, leave=False)
        for batch_idx, batch in enumerate(pbar):
            images  = batch["image"].to(device, non_blocking=True)
            targets = batch["mask"].to(device, non_blocking=True)

            if train:
                optimizer.zero_grad(set_to_none=True)

            amp_on = MIXED_PRECISION and device.type == "cuda"
            with autocast(device_type="cuda", enabled=amp_on):
                logits = model(images)
                loss   = criterion(logits, targets)

            if train:
                if scaler and amp_on:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                    optimizer.step()

            with torch.no_grad():
                metrics.update((torch.sigmoid(logits) >= BINARY_THRESHOLD).float(), targets)

            total_loss += float(loss.item()); n_batches += 1
            pbar.set_postfix(loss=f"{loss.item():.4f}")

            if print_mem_at_batch_1 and batch_idx == 0 and device.type == "cuda":
                gb = torch.cuda.memory_allocated(device) / 1024**3
                print(f"  [memory] after batch 1: {gb:.2f} GB used.")

    return total_loss / max(n_batches, 1), metrics.compute()

print("OK")

OK


In [11]:
# ── Cell 9: Checkpoint Helpers (IDENTICAL) ────────────────────────────────────
def save_checkpoint(path, model, optimizer, scheduler, epoch, best_val_iou, history):
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save({"epoch": epoch, "model": model.state_dict(),
                "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                "best_val_iou": best_val_iou, "history": history,
                "rng": get_rng_state()}, tmp)
    os.replace(tmp, path)

def try_load_checkpoint(path, model, optimizer, scheduler, device):
    if not path.exists():
        return None
    try:
        ckpt = torch.load(path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        if "rng" in ckpt:
            try: set_rng_state(ckpt["rng"])
            except Exception as e: print(f"  [warn] RNG restore failed: {e}")
        return {"epoch": ckpt["epoch"], "best_val_iou": ckpt["best_val_iou"],
                "history": ckpt.get("history", [])}
    except Exception as e:
        print(f"  [warn] checkpoint unreadable ({e}). Starting fresh.")
        return None

def append_metrics_json(path, history):
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w") as f: json.dump(history, f, indent=2)
    os.replace(tmp, path)

print("OK")

OK


In [12]:
# ── Cell 10: Plotting (IDENTICAL) ────────────────────────────────────────────
def plot_training_curves(history, out_path):
    if not history: return
    epochs = [h["epoch"] for h in history]
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    axes[0,0].plot(epochs,[h["train_loss"] for h in history],label="train",marker="o")
    axes[0,0].plot(epochs,[h["val_loss"]   for h in history],label="val",  marker="s")
    axes[0,0].set_title("Loss"); axes[0,0].legend(); axes[0,0].grid(alpha=0.3)
    axes[0,1].plot(epochs,[h["train_iou"] for h in history],label="train",marker="o")
    axes[0,1].plot(epochs,[h["val_iou"]   for h in history],label="val",  marker="s")
    axes[0,1].set_title("IoU (Jaccard)"); axes[0,1].legend(); axes[0,1].grid(alpha=0.3)
    axes[1,0].plot(epochs,[h["train_f1"] for h in history],label="train",marker="o")
    axes[1,0].plot(epochs,[h["val_f1"]   for h in history],label="val",  marker="s")
    axes[1,0].set_title("F1 Score"); axes[1,0].legend(); axes[1,0].grid(alpha=0.3)
    axes[1,1].plot(epochs,[h["lr"] for h in history],marker="o",color="tab:green")
    axes[1,1].set_title("Learning Rate"); axes[1,1].set_yscale("log"); axes[1,1].grid(alpha=0.3)
    fig.suptitle("ForensicNet (ConvNeXt-L) — TGIF subset", fontsize=14)
    fig.tight_layout(); fig.savefig(out_path, dpi=120, bbox_inches="tight"); plt.close(fig)

def plot_confusion_matrix(metrics, out_path):
    tp,fp,tn,fn = metrics["tp"],metrics["fp"],metrics["tn"],metrics["fn"]
    cm = np.array([[tn,fp],[fn,tp]], dtype=np.int64); total = cm.sum()+1e-8
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(["Pred: Real (0)","Pred: Fake (1)"])
    ax.set_yticklabels(["True: Real (0)","True: Fake (1)"])
    ax.set_title("Pixel-level Confusion Matrix (Test Set)")
    for i in range(2):
        for j in range(2):
            count = cm[i,j]; pct = 100.0*count/total
            ax.text(j,i,f"{count:,}\n({pct:.2f}%)",ha="center",va="center",
                    color="white" if count>cm.max()/2 else "black",fontsize=11)
    fig.colorbar(im,ax=ax,fraction=0.046,pad=0.04)
    fig.tight_layout(); fig.savefig(out_path,dpi=120,bbox_inches="tight"); plt.close(fig)

print("OK")

OK


In [13]:
# ── Cell 11: Build Model + Optimizer + Scheduler (IDENTICAL settings) ─────────
print("[3/5] Building model ...")
model = ForensicNet(image_size=IMAGE_SIZE, pretrained=True).to(device)

total_params  = sum(p.numel() for p in model.parameters())
model_size_mb = total_params * 4 / 1024**2
print(f"   Parameters : {total_params/1e6:.2f} M  |  Size : {model_size_mb:.1f} MB")

criterion = BCEDiceLoss(pos_weight=POS_WEIGHT).to(device)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=LR_FACTOR,
                               patience=LR_PATIENCE, min_lr=LR_MIN)
scaler    = GradScaler(enabled=MIXED_PRECISION and device.type == "cuda")

latest_ckpt  = OUTPUT_DIR / "forensicnet_checkpoint_latest.pth"
best_ckpt    = OUTPUT_DIR / "forensicnet_checkpoint_best.pth"
metrics_json = OUTPUT_DIR / "forensicnet_metrics_history.json"

start_epoch = 1; best_val_iou = -1.0; history = []; epochs_no_improve = 0

loaded = try_load_checkpoint(latest_ckpt, model, optimizer, scheduler, device)
if loaded:
    start_epoch  = loaded["epoch"] + 1
    best_val_iou = loaded["best_val_iou"]
    history      = loaded["history"]
    print(f"\n[resume] Resuming from epoch {start_epoch} (best val IoU: {best_val_iou:.4f})")
else:
    print("\n[resume] Starting fresh.")

[3/5] Building model ...


model.safetensors:   0%|          | 0.00/791M [00:00<?, ?B/s]

   Parameters : 199.56 M  |  Size : 761.3 MB

[resume] Starting fresh.


In [14]:
import os, glob

# আগের MVSSNet++ এর files delete করো
for f in glob.glob("/kaggle/working/*.pth") + glob.glob("/kaggle/working/*.zip"):
    os.remove(f)
    print(f"Deleted: {f}")

# কতটুকু space আছে দেখো
os.system("df -h /kaggle/working")

Deleted: /kaggle/working/tgif-subset.zip
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  9.0G   11G  46% /kaggle/working


0

In [15]:
# ── Cell 12: Training (IDENTICAL loop) ───────────────────────────────────────
print(f"[4/5] Training for up to {EPOCHS} epochs ...")
overall_start = time.time(); epoch_times = []

for epoch in range(start_epoch, EPOCHS + 1):
    epoch_start = time.time()
    print(f"\n── Epoch {epoch}/{EPOCHS} ──")

    try:
        train_loss, train_metrics = run_one_epoch(
            model, train_loader, criterion, optimizer, scaler,
            device, train=True, desc=f"train {epoch}",
            print_mem_at_batch_1=(epoch == start_epoch),
        )
    except torch.cuda.OutOfMemoryError:
        print(f"\nERROR — Out of GPU memory! Try BATCH_SIZE=8")
        torch.cuda.empty_cache(); break

    val_loss, val_metrics = run_one_epoch(
        model, val_loader, criterion, None, None,
        device, train=False, desc=f"val {epoch}",
    )

    current_lr = optimizer.param_groups[0]["lr"]
    epoch_secs = time.time() - epoch_start
    epoch_times.append(epoch_secs)
    scheduler.step(val_metrics["iou"])

    print(f"  train loss {train_loss:.4f} | val loss {val_loss:.4f} | "
          f"val IoU {val_metrics['iou']:.4f} | val F1 {val_metrics['f1']:.4f} | "
          f"val Prec {val_metrics['precision']:.4f} | val Rec {val_metrics['recall']:.4f} | "
          f"lr {current_lr:.2e} | {epoch_secs:.0f}s")

    history.append({
        "epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
        "train_iou": train_metrics["iou"], "val_iou": val_metrics["iou"],
        "train_f1":  train_metrics["f1"],  "val_f1":  val_metrics["f1"],
        "val_precision": val_metrics["precision"], "val_recall": val_metrics["recall"],
        "val_dice": val_metrics["dice"], "val_accuracy": val_metrics["pixel_accuracy"],
        "lr": current_lr, "epoch_seconds": epoch_secs,
    })

    save_checkpoint(latest_ckpt, model, optimizer, scheduler, epoch, best_val_iou, history)
    append_metrics_json(metrics_json, history)

    if val_metrics["iou"] > best_val_iou:
        best_val_iou = val_metrics["iou"]; epochs_no_improve = 0
        save_checkpoint(best_ckpt, model, optimizer, scheduler, epoch, best_val_iou, history)
        print(f"  OK Best val IoU = {best_val_iou:.4f} — checkpoint saved!")
    else:
        epochs_no_improve += 1
        print(f"  No improvement for {epochs_no_improve} epoch(s). "
              f"Best: {best_val_iou:.4f} (early stop at {EARLY_STOP_PATIENCE})")

    if epochs_no_improve >= EARLY_STOP_PATIENCE:
        print(f"\n[early stop] Stopping.")
        break

total_train_time = time.time() - overall_start
print(f"\n  Done. Total: {total_train_time:.0f}s  (avg {np.mean(epoch_times):.0f}s/epoch)")

[4/5] Training for up to 30 epochs ...

── Epoch 1/30 ──


train 1:   0%|          | 0/473 [00:00<?, ?it/s]

  [memory] after batch 1: 3.06 GB used.


val 1:   0%|          | 0/64 [00:00<?, ?it/s]

  train loss 0.7478 | val loss 0.7076 | val IoU 0.2578 | val F1 0.4100 | val Prec 0.2591 | val Rec 0.9815 | lr 1.00e-04 | 589s
  OK Best val IoU = 0.2578 — checkpoint saved!

── Epoch 2/30 ──


...

val 24:   0%|          | 0/64 [00:00<?, ?it/s]

  train loss 0.1593 | val loss 0.2946 | val IoU 0.8354 | val F1 0.9103 | val Prec 0.9428 | val Rec 0.8801 | lr 1.25e-05 | 563s
  No improvement for 6 epoch(s). Best: 0.8441 (early stop at 7)

── Epoch 25/30 ──


train 25:   0%|          | 0/473 [00:00<?, ?it/s]

val 25:   0%|          | 0/64 [00:00<?, ?it/s]

  train loss 0.1581 | val loss 0.2977 | val IoU 0.8357 | val F1 0.9105 | val Prec 0.9441 | val Rec 0.8792 | lr 1.25e-05 | 563s
  No improvement for 7 epoch(s). Best: 0.8441 (early stop at 7)

[early stop] Stopping.

  Done. Total: 15138s  (avg 586s/epoch)


In [16]:
# ── Cell 13: Test Evaluation & Results (IDENTICAL format) ────────────────────
print("[5/5] Evaluating on TEST set (best checkpoint) ...")
if best_ckpt.exists():
    ckpt = torch.load(best_ckpt, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model"])
    print(f"   Loaded best checkpoint: epoch {ckpt['epoch']} "
          f"(val IoU = {ckpt['best_val_iou']:.4f})")
else:
    print("   No best checkpoint — using current weights.")

test_loss, test_metrics = run_one_epoch(
    model, test_loader, criterion, None, None,
    device, train=False, desc="test",
)

final_lr       = optimizer.param_groups[0]["lr"]
epochs_trained = history[-1]["epoch"] if history else 0

results_text = (
    "======================================================\n"
    "   Results — ForensicNet (ConvNeXt-L)  [test]\n"
    "======================================================\n"
    f"  Pixel Accuracy     : {test_metrics['pixel_accuracy']:.4f}\n"
    f"  Precision          : {test_metrics['precision']:.4f}\n"
    f"  Recall             : {test_metrics['recall']:.4f}\n"
    f"  F1 Score           : {test_metrics['f1']:.4f}\n"
    f"  IoU (Jaccard)      : {test_metrics['iou']:.4f}\n"
    f"  Dice Score         : {test_metrics['dice']:.4f}\n"
    "  ─────────────────────────────────\n"
    f"  Model Size         : {model_size_mb:.1f} MB\n"
    f"  Total Params       : {total_params/1e6:.2f} M\n"
    f"  Total Train Time   : {total_train_time:.0f} s\n"
    f"  Avg Time / Epoch   : {np.mean(epoch_times) if epoch_times else 0:.0f} s\n"
    f"  Final LR           : {final_lr:.2e}\n"
    f"  Epochs Trained     : {epochs_trained}\n"
    f"  POS_WEIGHT used    : {POS_WEIGHT}\n"
    "======================================================\n"
)

(OUTPUT_DIR / "forensicnet_final_results.txt").write_text(results_text)
print("\n" + results_text)

plot_training_curves(history, OUTPUT_DIR / "forensicnet_training_curves.png")
plot_confusion_matrix(test_metrics, OUTPUT_DIR / "forensicnet_confusion_matrix.png")

print("Saved:")
print("  forensicnet_final_results.txt")
print("  forensicnet_training_curves.png")
print("  forensicnet_confusion_matrix.png")
print("  forensicnet_metrics_history.json")
print("  forensicnet_checkpoint_best.pth")
print("\nDownload from Output tab before session ends!")

[5/5] Evaluating on TEST set (best checkpoint) ...
   Loaded best checkpoint: epoch 18 (val IoU = 0.8441)


test:   0%|          | 0/65 [00:00<?, ?it/s]


   Results — ForensicNet (ConvNeXt-L)  [test]
  Pixel Accuracy     : 0.9931
  Precision          : 0.9252
  Recall             : 0.9108
  F1 Score           : 0.9179
  IoU (Jaccard)      : 0.8483
  Dice Score         : 0.9179
  ─────────────────────────────────
  Model Size         : 761.3 MB
  Total Params       : 199.56 M
  Total Train Time   : 15138 s
  Avg Time / Epoch   : 586 s
  Final LR           : 1.25e-05
  Epochs Trained     : 25
  POS_WEIGHT used    : 3.0

Saved:
  forensicnet_final_results.txt
  forensicnet_training_curves.png
  forensicnet_confusion_matrix.png
  forensicnet_metrics_history.json
  forensicnet_checkpoint_best.pth

Download from Output tab before session ends!
